# 02 Reproducibility of the evaluation

Measures where the prediction differences found in notebook 01 come from, using the seed-42
item-level adapter that produced the released files (SHA-256 prefix `1de40b6b`).

| Section | Content | GPU |
|---|---|---|
| [1] | SocialIQa seed-45 cross-tabulation of the mismatches from notebook 01 | no |
| [2] | the same evaluation twice in one session, with one model object | yes |
| [3] | reload in the same session (PeftModel) and loading with unsloth `FastLanguageModel.from_pretrained` | yes |
| [4] | mismatch summary and letter-margin analysis | no |

Evaluation is greedy and one item at a time (no padding), as in the main pipeline. [1] can be run
before the GPU setup. If the session is interrupted during [2], delete both run1 and run2 files
and run [2] again, since the two runs must share one session.

`DRIVE_DIR` is the Drive folder used by `notebooks/tombench_slm_qlora_complete_pipeline.ipynb`;
`tier4_v2_fixed/` inside it holds that notebook's outputs (split and manifest files, per-item
CSVs, adapters). The other folders are created by the revision notebooks; see
`notebooks/revision/README.md` for where their files are in `results/`.

## [1] SocialIQa seed 45 cross-tabulation (no GPU needed)

In [ ]:
# -- [1] Reads the CSVs saved by notebook 01 only; no GPU or model needed --
from google.colab import drive
drive.mount('/content/drive')
import os, pandas as pd

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
OUT_DIR_P2  = f'{DRIVE_DIR}/cyclic_shift_check'
OUT_DIR_R   = f'{DRIVE_DIR}/repro_check'
os.makedirs(OUT_DIR_R, exist_ok=True)

def flip_table(bm, s):
    old = pd.read_csv(f'{RESULTS_DIR}/finetuned_seed{s}_{bm}.csv')
    new = pd.read_csv(f'{OUT_DIR_P2}/plain_seed{s}_{bm}.csv')
    m = new.merge(old, on='item_id', suffixes=('_new', '_old'))
    return m, m[m['pred_new'] != m['pred_old']]

m45, d45 = flip_table('SocialIQa', 45)
print(f'SocialIQa seed45: mismatches {len(d45)}/{len(m45)}')

ct = pd.crosstab(d45['pred_old'], d45['pred_new'], margins=True)
ct.index.name, ct.columns.name = 'released_pred', 'reeval_pred'
print('\n[Cross-tabulation] rows = released prediction, columns = re-evaluation')
print(ct)
ct.to_csv(f'{OUT_DIR_R}/socialiqa_seed45_flip_crosstab.csv')

# correct/incorrect transitions behind the net change
trans = pd.crosstab(d45['correct_old'], d45['correct_new'])
trans.index.name, trans.columns.name = 'released_correct', 'reeval_correct'
print('\n[Correct/incorrect transitions]')
print(trans)
n_lost = int(((d45['correct_old'] == 1) & (d45['correct_new'] == 0)).sum())
n_gain = int(((d45['correct_old'] == 0) & (d45['correct_new'] == 1)).sum())
print(f'correct->wrong {n_lost}, wrong->correct {n_gain}, net {n_gain - n_lost} items = {(n_gain - n_lost) / len(m45) * 100:+.2f}pp')

# gold letters of the mismatching items
print('\n[Gold letter of mismatching items]')
print(d45['answer_new'].value_counts().sort_index())

# reference: share of the most common flip direction for every seed
rows = []
for s in [42, 43, 44, 45, 46]:
    _, d = flip_table('SocialIQa', s)
    pair = (d['pred_old'] + '->' + d['pred_new']).value_counts()
    rows.append({'seed': s, 'n_flip': len(d), 'top_direction': pair.index[0] if len(pair) else None,
                 'top_share_pct': pair.iloc[0] / len(d) * 100 if len(d) else None,
                 'net_correct': int(d['correct_new'].sum() - d['correct_old'].sum())})
cmp = pd.DataFrame(rows)
print('\n[SocialIQa by seed] most common flip direction and its share')
print(cmp.to_string(index=False))
cmp.to_csv(f'{OUT_DIR_R}/socialiqa_flip_direction_by_seed.csv', index=False)

## [0] Setup (GPU from here on)

Run the cells in order: [0-1] installs the pinned package versions; restart the runtime
(Runtime > Restart session) and continue from [0-2]. Results are written to Google Drive, and
cells that find their output already saved skip it, so an interrupted run can be resumed by
running [0-2] again and then the interrupted cell.

In [ ]:
# -- [0-1] Install packages (versions pinned to requirements.lock.txt) --
import sys, os, time
import torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB VRAM)')
t0 = time.time()
get_ipython().system('pip install -q unsloth==2026.8.18 unsloth_zoo==2026.8.12')
get_ipython().system('pip install -q transformers==5.5.0 trl==0.24.0 accelerate==1.14.0 bitsandbytes==0.50.1 peft==0.20.0')
get_ipython().system('pip install -q "datasets==3.6.0" tqdm')
print(f'   {time.time()-t0:.0f}s  ->  restart the runtime, then run from [0-2]')

In [ ]:
# -- [0-2] Mount Drive, set paths, helpers (as in notebook 01, plus the letter margin) --
from google.colab import drive
drive.mount('/content/drive')

import os, re, json as _json, time, string, hashlib, platform
import pandas as pd, numpy as np, torch
import transformers, peft, unsloth
from unsloth import FastLanguageModel
from peft import PeftModel
assert transformers.__version__ == '5.5.0', f'transformers version not pinned: {transformers.__version__}'

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
OUT_DIR_P2  = f'{DRIVE_DIR}/cyclic_shift_check'
OUT_DIR_R   = f'{DRIVE_DIR}/repro_check'
os.makedirs(OUT_DIR_R, exist_ok=True)

BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
ADAPTER_42 = f'{RESULTS_DIR}/adapter_seed42'

# confirm that the adapter is the one that produced the released CSVs
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()[:16]
_h = sha16(f'{ADAPTER_42}/adapter_model.safetensors')
print('adapter_seed42 sha256:', _h)
assert _h == '1de40b6b5c101d83', 'not the released seed-42 adapter (tier4_v2_fixed/adapter_seed42)'

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)
ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')

def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def load_manifest(bm):
    path = f'{RESULTS_DIR}/split_test_full_v2.json' if bm == 'ToMBench' else f'{RESULTS_DIR}/manifest_{bm}.json'
    with open(path, encoding='utf-8') as f:
        return _json.load(f)

MANIFESTS = {bm: load_manifest(bm) for bm in BENCHMARKS}
print({bm: len(v) for bm, v in MANIFESTS.items()})

@torch.no_grad()
def evaluate_margin(eval_records, model, tokenizer, desc):
    # Same input and greedy decoding as evaluate() (one item at a time, no padding).
    # Also records, at the step that emits the answer letter, the gap between the
    # top two A-E letter logits (margin); a small margin marks a near-tie item.
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    letter_ids = {}
    for L in 'ABCDE':
        ids = tokenizer.encode(L, add_special_tokens=False)
        assert len(ids) == 1, f'{L} is not a single token: {ids}'
        letter_ids[ids[0]] = L
    lid = torch.tensor(list(letter_ids.keys()))
    rows = []
    for r in tqdm(eval_records, desc=desc):
        inputs = tokenizer(to_text(tokenizer, r['user_prompt']), return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id,
                             output_scores=True, return_dict_in_generate=True)
        gen_ids = out.sequences[0][inputs['input_ids'].shape[1]:]
        gen = tokenizer.decode(gen_ids, skip_special_tokens=True)
        pred = extract_letter(gen)
        margin, top2 = None, None
        for step, tok in enumerate(gen_ids.tolist()):
            if tok in letter_ids and step < len(out.scores):
                sc = out.scores[step][0].float().cpu()[lid]
                v, i = torch.sort(sc, descending=True)
                margin = float(v[0] - v[1])
                top2 = letter_ids[lid[i[1]].item()]
                break
        rows.append({'item_id': r['item_id'], 'answer': r['answer'], 'pred': pred, 'raw': gen,
                     'correct': int(pred == r['answer']), 'letter_margin': margin, 'runner_up': top2})
    return pd.DataFrame(rows)

def run_pass(tag, model, tokenizer, benchmarks=BENCHMARKS):
    # saved as repro_<tag>_seed42_<bm>.csv; skipped if present
    for bm in benchmarks:
        p = f'{OUT_DIR_R}/repro_{tag}_seed42_{bm}.csv'
        if os.path.exists(p):
            print(f'[skip] {tag} {bm}'); continue
        t0 = time.time()
        df = evaluate_margin(MANIFESTS[bm], model, tokenizer, desc=f'{tag}-{bm}')
        df.to_csv(p, index=False)
        print(f'[{tag}] {bm}: acc={df["correct"].mean()*100:.2f}%  ({(time.time()-t0)/60:.1f} min)')

def load_peft_path():
    # same loading path as notebook 01 (PeftModel.from_pretrained)
    base, tok = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    model = PeftModel.from_pretrained(base, ADAPTER_42)
    FastLanguageModel.for_inference(model)
    return model, tok

# record the session environment
meta = {'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda, 'torch': torch.__version__,
        'transformers': transformers.__version__, 'peft': peft.__version__, 'unsloth': unsloth.__version__,
        'python': platform.python_version(), 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(f'{OUT_DIR_R}/session_meta.json', 'w') as f:
    _json.dump(meta, f, indent=2)
print(meta)

## [2] The same evaluation twice in one session

Loads the seed-42 adapter once with `PeftModel.from_pretrained` (as in notebook 01) and evaluates all
4,842 items twice without changing anything. No mismatch between run1 and run2 means that the
evaluation within a session is deterministic.

In [ ]:
# -- [2] Load the model once, then evaluate twice in a row (run1, run2) --
model_a, tok_a = load_peft_path()
run_pass('run1', model_a, tok_a)
run_pass('run2', model_a, tok_a)

## [3] Loading path

The released CSVs were produced by the model in memory right after training (LoRA layers attached
by unsloth `get_peft_model`), whereas notebook 01 reattached the saved adapter with
`PeftModel.from_pretrained`.

- [3-A] reload: delete the model from [2] and load it again the same way
- [3-B] unsloth: load the adapter folder with `FastLanguageModel.from_pretrained`

Only Hi-ToM and SocialIQa, which had the most mismatches, are run here (`BM_3`); notebook 03
covers the rest.

In [ ]:
# -- [3-A] Reload in the same session, then evaluate --
BM_3 = ['HiToM', 'SocialIQa']
del model_a; torch.cuda.empty_cache()
model_b, tok_b = load_peft_path()
run_pass('reload', model_b, tok_b, BM_3)
del model_b; torch.cuda.empty_cache()

In [ ]:
# -- [3-B] Load the adapter with unsloth, then evaluate --
model_c, tok_c = FastLanguageModel.from_pretrained(model_name=ADAPTER_42, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
FastLanguageModel.for_inference(model_c)
print(type(model_c).__name__)
run_pass('unslothload', model_c, tok_c, BM_3)
del model_c; torch.cuda.empty_cache()

## [4] Summary (no GPU needed, reads the CSVs only)

Pairs compared: run1 vs run2 (same session and model object), run1 vs reload, run1 vs `gate1011`
(notebook 01, same code and loading path, different session), run1 vs the released CSVs, and the
unsloth loading vs the released CSVs. The last part checks whether the flipped items have small
letter margins.

In [ ]:
# -- [4] Mismatch summary and margin analysis --
from google.colab import drive
drive.mount('/content/drive')
import os, json as _json, pandas as pd, numpy as np
DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
OUT_DIR_P2  = f'{DRIVE_DIR}/cyclic_shift_check'
OUT_DIR_R   = f'{DRIVE_DIR}/repro_check'
BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']

def src(name, bm):
    return {'run1':        f'{OUT_DIR_R}/repro_run1_seed42_{bm}.csv',
            'run2':        f'{OUT_DIR_R}/repro_run2_seed42_{bm}.csv',
            'reload':      f'{OUT_DIR_R}/repro_reload_seed42_{bm}.csv',
            'unslothload': f'{OUT_DIR_R}/repro_unslothload_seed42_{bm}.csv',
            'gate1011':    f'{OUT_DIR_P2}/plain_seed42_{bm}.csv',
            'published':   f'{RESULTS_DIR}/finetuned_seed42_{bm}.csv'}[name]

PAIRS = [('run1', 'run2'), ('run1', 'reload'), ('run1', 'gate1011'), ('run1', 'published'),
         ('unslothload', 'published'), ('gate1011', 'published')]
rows = []
for a, b in PAIRS:
    for bm in BENCHMARKS:
        pa, pb = src(a, bm), src(b, bm)
        if not (os.path.exists(pa) and os.path.exists(pb)): continue
        m = pd.read_csv(pa).merge(pd.read_csv(pb), on='item_id', suffixes=('_a', '_b'))
        acc_a, acc_b = m['correct_a'].mean() * 100, m['correct_b'].mean() * 100
        rows.append({'pair': f'{a} vs {b}', 'benchmark': bm, 'n': len(m),
                     'mismatch': int((m['pred_a'] != m['pred_b']).sum()),
                     'acc_a': round(acc_a, 2), 'acc_b': round(acc_b, 2), 'acc_diff_pp': round(acc_a - acc_b, 2)})
summ = pd.DataFrame(rows)
summ.to_csv(f'{OUT_DIR_R}/repro_mismatch_summary.csv', index=False)
print(summ.to_string(index=False))

tot = summ.groupby('pair', sort=False)[['mismatch', 'n']].sum()
tot['rate_pct'] = (tot['mismatch'] / tot['n'] * 100).round(2)
print('\n[Totals by pair]')
print(tot)
tot.to_csv(f'{OUT_DIR_R}/repro_mismatch_totals.csv')

# margin: run1 items that differ from the released predictions vs items that do not
mr = []
for bm in BENCHMARKS:
    p1, pp = src('run1', bm), src('published', bm)
    if not (os.path.exists(p1) and os.path.exists(pp)): continue
    m = pd.read_csv(p1).merge(pd.read_csv(pp)[['item_id', 'pred']].rename(columns={'pred': 'pred_pub'}), on='item_id')
    m['flipped'] = m['pred'] != m['pred_pub']
    m['benchmark'] = bm
    mr.append(m)
if mr:
    mm = pd.concat(mr)
    print('\n[letter margin] items that differ from the released predictions vs items that do not (logit gap; small = near tie)')
    print(mm.groupby('flipped')['letter_margin'].describe()[['count', 'mean', '50%', 'max']])
    for th in [0.25, 0.5, 1.0]:
        sub = mm[mm['letter_margin'] < th]
        print(f'margin < {th}: {len(sub)/len(mm)*100:.1f}% of items; of the flipped items, '
              f'{(mm["flipped"] & (mm["letter_margin"] < th)).sum()}/{mm["flipped"].sum()}')
    if mm['flipped'].any():
        print('share of flipped items whose runner-up is the released prediction:',
              f'{(mm.loc[mm["flipped"], "runner_up"] == mm.loc[mm["flipped"], "pred_pub"]).mean()*100:.1f}%')
    mm.to_csv(f'{OUT_DIR_R}/repro_margin_run1_vs_published.csv', index=False)

meta_p = f'{OUT_DIR_R}/session_meta.json'
if os.path.exists(meta_p):
    print('\nSession environment:', _json.load(open(meta_p)))